In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')
import lib_dna_member.job_manager as job_manager
import lib_dna_member.acquisition_features as features
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient
from lib.s3 import input_data_validator, _init_dbutils_spark

In [0]:
%run ../../config/utils

In [0]:
def integrate_acquisition(job):
    """
    Integrate acquisition variables for the given population
    Parameters:
        job (object): Job Manager object based on the current config file

    Returns:
        dna (pyspark.sql.DataFrame): Acquisition variable added to the given
                                     population
    """
    print("IN INTEGRATE ACQ ")

    dna = job.tables["cubes_member_features"]
    orig_cols = dna.columns

    dna = features.feature_age_income(job, dna)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna

In [0]:
job = job_manager.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

In [0]:
recency_lookback_duration = job.config["params"].get(
    "recency_lookback_duration", {}
)
member_dna_input_data_validator(
    fs_cubes_member, silver_master_member_basic,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)

_init_dbutils_spark(spark)
input_data_validator(recency_lookback_duration.get("local", 0).get('acq_dna_path',50), aqc_member_dna_path)

In [0]:
job.read_table("cubes_member_features") 
job.tables['acq_dna'] = spark.read.parquet(aqc_member_dna_path) # TODO: TEMP
job.read_table("member_basic")

In [0]:
features = integrate_acquisition(job)

### Save results

In [0]:
spark.sql(f"DELETE FROM {fs_cubes_acquisition}")

fe = FeatureEngineeringClient()

fe.write_table(
    name=fs_cubes_acquisition,
    df=features,
    mode="merge"
)